# Embedding Collapse Diagnostics

This notebook mirrors `visualizations.ssl4eo.embedding_collapse_diagnostics` using its original entry point.

1. Configure the arguments below.
2. Set `RUN_PIPELINE = True`.
3. Run the execution cell to launch the full diagnostic export.

In [1]:
import sys
from pathlib import Path

# Ensure the open_clip_train helpers are importable when running inside the notebook.
repo_root = Path.cwd()
sys.path.insert(0, str(repo_root.parent.parent.parent))

from visualizations.ssl4eo import embedding_collapse_diagnostics as ecd
from IPython.display import Image, display\

import matplotlib.pyplot as plt
# plt.switch_backend("module://matplotlib_inline.backend_inline")

/home/juro4948/miniconda3/envs/ciip/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# --- Required arguments -------------------------------------------------
model_root = '/local/ms-data/SSL4EO/model/'
model_path = '2025_09_11-14_15_30-model_resnet50-lr_0.0005-b_128-j_6-p_amp/'
# '2025_09_11-14_15_30-model_resnet50-lr_0.0005-b_128-j_6-p_amp/'
# '2025_11_05-21_04_44-model_resnet50-lr_0.001-b_128-j_6-p_amp/'
checkpoint_root = Path(model_root) / model_path / "checkpoints"
output_dir = Path("diagnostics/output")

# --- Optional arguments -------------------------------------------------
config_name = "prod_default"
config_path = None  # Path or None
dataset_root = None  # Path or None
subset_size = 2048
subset_seed = 42
checkpoint_pattern = r"epoch"
max_checkpoints = 20
include_init = False
device_override = None  # e.g. "cuda:0" or "cpu"
skip_final_fc = False
use_orthogonal_mapping = False
negative_samples = 2000
vc_gamma = None
random_seed = 42
spectrum_top_k = 800
cca_top_k = 5
linear_probe_csv = None  # Path or None
linear_probe_pattern = None
linear_probe_k = None
linear_probe_metric = "accuracy"
tsne_samples = 500
umap_samples = 500
cosine_hist_bins = 50

In [3]:
DISPLAY_NOTEBOOK_OUTPUTS = True
SAVE_NOTEBOOK_OUTPUTS = True
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".gif")


def display_generated_images(display_dir: Path, *, image_extensions=(".png", ".jpg", ".jpeg", ".gif")):
    """Render PNG/JPEG artifacts generated by the diagnostics pipeline."""
    if not display_dir.exists():
        print(f"No outputs available because {display_dir} does not exist.")
        return []

    image_paths = sorted(
        path for path in display_dir.rglob('*')
        if path.suffix.lower() in image_extensions
    )

    if not image_paths:
        print(f"No image outputs found in {display_dir}.")
        return []

    for image_path in image_paths:
        print(f"Found image: {image_path.relative_to(display_dir)}")

    return image_paths

In [4]:
DISPLAY_NOTEBOOK_OUTPUTS = True
SAVE_NOTEBOOK_OUTPUTS = True
RUN_PIPELINE = True

if RUN_PIPELINE:
    from tempfile import TemporaryDirectory
    temp_dir_obj = None
    run_output_dir = output_dir

    if not SAVE_NOTEBOOK_OUTPUTS:
        temp_dir_obj = TemporaryDirectory(prefix="embedding_diagnostics_")
        run_output_dir = Path(temp_dir_obj.name)

    argv = [
        "embedding_collapse_diagnostics",
        "--checkpoint-root", str(checkpoint_root),
        "--output-dir", str(run_output_dir),
        "--config-name", config_name,
        "--subset-size", str(subset_size),
        "--subset-seed", str(subset_seed),
        "--checkpoint-pattern", checkpoint_pattern,
        "--negative-samples", str(negative_samples),
        "--random-seed", str(random_seed),
        "--spectrum-top-k", str(spectrum_top_k),
        "--cca-top-k", str(cca_top_k),
        "--tsne-samples", str(tsne_samples),
        "--umap-samples", str(umap_samples),
        "--cosine-hist-bins", str(cosine_hist_bins),
        "--linear-probe-metric", linear_probe_metric,
    ]

    if config_path is not None:
        argv.extend(["--config-path", str(config_path)])
    if dataset_root is not None:
        argv.extend(["--dataset-root", str(dataset_root)])
    if max_checkpoints is not None:
        argv.extend(["--max-checkpoints", str(max_checkpoints)])
    if device_override is not None:
        argv.extend(["--device", device_override])
    if vc_gamma is not None:
        argv.extend(["--vc-gamma", str(vc_gamma)])
    if linear_probe_csv is not None:
        argv.extend(["--linear-probe-csv", str(linear_probe_csv)])
    if linear_probe_pattern is not None:
        argv.extend(["--linear-probe-pattern", linear_probe_pattern])
    if linear_probe_k is not None:
        argv.extend(["--linear-probe-k", str(linear_probe_k)])

    if include_init:
        argv.append("--include-init")
    if skip_final_fc:
        argv.append("--skip-final-fc")
    if use_orthogonal_mapping:
        argv.append("--use-orthogonal-mapping")

    previous_argv = sys.argv[:]
    try:
        sys.argv = argv
        ecd.main()
    finally:
        sys.argv = previous_argv


    if DISPLAY_NOTEBOOK_OUTPUTS:
        display_generated_images(run_output_dir)

    if SAVE_NOTEBOOK_OUTPUTS:
        print(f"Outputs saved to {run_output_dir.resolve()}")
    else:
        print("Outputs generated in a temporary directory and not saved to disk.")
        if temp_dir_obj is not None:
            temp_dir_obj.cleanup()
else:
    print("Update the configuration cell above and set RUN_PIPELINE = True to execute.")




INFO:embedding_collapse:Loading config from /home/juro4948/ciip/ciip/open_clip_train/configs/prod_default.yaml
INFO:embedding_collapse:Using 2048 samples per epoch from dataset root /local/ms-data/SSL4EO/
INFO:embedding_collapse:Running extraction on cuda with input dtype torch.float16
INFO:embedding_collapse:Extracting embeddings for epoch_20.pt (epoch index 0)


Number of locations in dataset: 251079
[1, 20]
Using hyperbolic model


INFO:root:Using ResNet50 for S1 without pretrained weights.
INFO:root:Using ResNet50 for S2 without pretrained weights.


Final - S1 Encoder Parameters:  29799424
Final - S2 Encoder Parameters:  29833920


INFO:embedding_collapse:Replaced incompatible layers: ['encoder_s1.fc', 'encoder_s2.fc']
/home/juro4948/ciip/visualizations/ssl4eo/embedding_collapse_diagnostics.py:671: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Extracting embeddings using LorentzCIIP model: True


ERROR:embedding_collapse:Skipping epoch_20.pt due to extraction failure: mat1 and mat2 shapes cannot be multiplied (1x1024 and 2048x1024)


RuntimeError: No embeddings were extracted from checkpoints in '/local/ms-data/SSL4EO/model/2025_09_11-14_15_30-model_resnet50-lr_0.0005-b_128-j_6-p_amp/checkpoints'